# Cluster smoke test: Customer 360 to Neo4j

A minimal end to end run that proves four things about the lab environment:

1. The cluster is attached and Spark runs a job.
2. `pip install` reaches PyPI from the driver.
3. Bolt port 7687 is open outbound to Aura.
4. A Spark result can be written to the graph and read back.

No Delta table, no Unity Catalog object, and no cluster library are needed. Everything
is generated in memory, so the notebook is safe to re-run and leaves nothing behind in
the workspace.

Fill in the four constants in the next code cell before running. Run the cells in order.

In [ ]:
%pip install neo4j

In [ ]:
# restartPython() clears every variable defined so far, so it has to happen before the
# constants below and not after.
dbutils.library.restartPython()

## 1. Connection details

Replace the four values with the ones from your Aura credentials file. Aura is TLS only,
so the URI scheme is `neo4j+s://`. A plain `neo4j://` or `bolt://` fails in the handshake,
which reads like a wrong password but is not one.

These are constants in notebook text on purpose, for a throwaway smoke test. Real workshop
content reads them from a secret scope with `dbutils.secrets.get`.

In [ ]:
NEO4J_URI = "neo4j+s://<instance-id>.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "<your-password>"
NEO4J_DATABASE = "neo4j"

# Size of the generated data set. Small on purpose: this is a smoke test, not a load test.
CUSTOMER_COUNT = 200
ORDER_COUNT = 2_000
SEED = 42

# Set to True and re-run the last cell to remove everything this notebook wrote.
RUN_CLEANUP = False

## 2. Is the cluster alive

The first Spark action is the real test. A cluster that is still starting, or that failed
to start, hangs or errors here rather than later.

In [ ]:
print(f"Spark version:  {spark.version}")
print(f"Parallelism:    {spark.sparkContext.defaultParallelism}")
print(f"Round trip:     {spark.range(1).count()} row")

## 3. Generate a random customer 360 data set

Two source frames, customers and orders, built from `spark.range` and `rand`. Each column
gets its own seed so the columns do not correlate with one another, and the seeds are fixed
so two runs produce the same data and the counts at the end are comparable.

In [ ]:
from pyspark.sql import Column, DataFrame, functions as F

FIRST_NAMES = ["Ada", "Bo", "Cleo", "Dev", "Esi", "Finn", "Gia", "Hank", "Ines", "Jun"]
LAST_NAMES = ["Alvarez", "Boateng", "Chen", "Dubois", "Eriksen", "Fischer", "Gupta", "Haddad"]
CITIES = ["Austin", "Berlin", "Cape Town", "Denver", "Lisbon", "Osaka", "Toronto"]
CHANNELS = ["web", "mobile", "partner", "in-store"]
PRODUCTS = ["Atlas Router", "Basalt Hub", "Cobalt Sensor", "Delta Gateway", "Ember Relay"]


def pick(values: list[str], seed: int) -> Column:
    """Pick one of `values` per row, uniformly, using a fixed seed.

    `element_at` is 1-based, hence the `+ 1`. Written as an array of literals rather than a
    join against a lookup frame so the whole generator stays a single narrow stage.
    """
    choices = F.array(*[F.lit(value) for value in values])
    return F.element_at(choices, (F.rand(seed) * len(values)).cast("int") + 1)


customers = (
    spark.range(1, CUSTOMER_COUNT + 1)
    .withColumnRenamed("id", "customer_id")
    .withColumn("first_name", pick(FIRST_NAMES, SEED + 1))
    .withColumn("last_name", pick(LAST_NAMES, SEED + 2))
    .withColumn("city", pick(CITIES, SEED + 3))
    .withColumn("signup_channel", pick(CHANNELS, SEED + 4))
    .withColumn("signup_date", F.expr(f"date_add(to_date('2024-01-01'), cast(rand({SEED + 5}) * 600 as int))"))
)

orders = (
    spark.range(1, ORDER_COUNT + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.rand(SEED + 6) * CUSTOMER_COUNT).cast("long") + 1)
    .withColumn("product", pick(PRODUCTS, SEED + 7))
    .withColumn("amount", F.round(F.rand(SEED + 8) * 480 + 20, 2))
    .withColumn("channel", pick(CHANNELS, SEED + 9))
    .withColumn("ordered_at", F.expr(f"date_add(to_date('2025-01-01'), cast(rand({SEED + 10}) * 365 as int))"))
    # `orders` is read three times below, by the rollup, the write, and the counts. `rand` is
    # deterministic per partition and per seed, so re-computing it would give the same values,
    # but caching removes the question and the two extra passes.
    .cache()
)

print(f"customers: {customers.count()}")
print(f"orders:    {orders.count()}")
display(orders.limit(5))

## 4. ETL: roll orders up into a customer view

One aggregation and one left join. The left join is what makes this a 360 view rather than
an order report: a customer who never ordered still gets a row, with a zeroed rollup and
the `prospect` tier.

In [ ]:
order_rollup = orders.groupBy("customer_id").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("amount"), 2).alias("lifetime_value"),
    F.max("ordered_at").alias("last_order_date"),
    F.countDistinct("product").alias("distinct_products"),
)

customer_360 = (
    customers.join(order_rollup, on="customer_id", how="left")
    .fillna({"order_count": 0, "lifetime_value": 0.0, "distinct_products": 0})
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .withColumn(
        "tier",
        F.when(F.col("lifetime_value") >= 3_000, "platinum")
        .when(F.col("lifetime_value") >= 1_500, "gold")
        .when(F.col("lifetime_value") > 0, "standard")
        .otherwise("prospect"),
    )
    .select(
        "customer_id",
        "full_name",
        "city",
        "signup_channel",
        "signup_date",
        "order_count",
        "lifetime_value",
        "last_order_date",
        "distinct_products",
        "tier",
    )
)

display(customer_360.orderBy(F.col("lifetime_value").desc()).limit(10))
display(customer_360.groupBy("tier").count().orderBy("tier"))

## 5. Can this cluster reach Aura

`verify_connectivity` fails three distinguishable ways, and the recovery differs for each:
a blocked port times out, a wrong password raises an authentication error, and a
non-TLS scheme fails during the handshake.

In [ ]:
from neo4j import GraphDatabase

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    driver.verify_connectivity()
    record = driver.execute_query(
        "RETURN 1 AS ok", database_=NEO4J_DATABASE
    ).records[0]

print(f"connected to {NEO4J_URI}, database {NEO4J_DATABASE}, RETURN 1 = {record['ok']}")

## 6. Load the graph

The Spark rows are collected to the driver and sent over Bolt in batched `UNWIND` writes.
That is the right shape at this size and it needs no cluster library. A production sized
job uses the Neo4j Connector for Apache Spark instead, which writes from the executors
and has to be installed on the cluster.

Every write is a `MERGE` on the key, so re-running the notebook updates rows rather than
duplicating them.

In [ ]:
from typing import Any

BATCH_SIZE = 500

CONSTRAINTS = [
    "CREATE CONSTRAINT customer_id IF NOT EXISTS FOR (c:Customer) REQUIRE c.customer_id IS UNIQUE",
    "CREATE CONSTRAINT order_id IF NOT EXISTS FOR (o:Order) REQUIRE o.order_id IS UNIQUE",
]

MERGE_CUSTOMERS = """
UNWIND $rows AS row
MERGE (c:Customer {customer_id: row.customer_id})
SET c.full_name         = row.full_name,
    c.city              = row.city,
    c.signup_channel    = row.signup_channel,
    c.signup_date       = row.signup_date,
    c.order_count       = row.order_count,
    c.lifetime_value    = row.lifetime_value,
    c.last_order_date   = row.last_order_date,
    c.distinct_products = row.distinct_products,
    c.tier              = row.tier
"""

MERGE_ORDERS = """
UNWIND $rows AS row
MERGE (o:Order {order_id: row.order_id})
SET o.product    = row.product,
    o.amount     = row.amount,
    o.channel    = row.channel,
    o.ordered_at = row.ordered_at
WITH o, row
MATCH (c:Customer {customer_id: row.customer_id})
MERGE (c)-[:PLACED]->(o)
"""


def to_rows(df: DataFrame) -> list[dict[str, Any]]:
    """Collect a DataFrame as plain dicts the driver can bind as a parameter.

    `datetime.date` values pass through as Neo4j dates, so no string formatting is needed.
    """
    return [row.asDict() for row in df.collect()]


def write_batches(query: str, rows: list[dict[str, Any]]) -> int:
    """Send `rows` through `query` in batches, and return how many were written."""
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        with driver.session(database=NEO4J_DATABASE) as session:
            for constraint in CONSTRAINTS:
                session.run(constraint)
            for start in range(0, len(rows), BATCH_SIZE):
                batch = rows[start : start + BATCH_SIZE]
                session.execute_write(lambda tx, b=batch: tx.run(query, rows=b).consume())
    return len(rows)


print(f"customers written: {write_batches(MERGE_CUSTOMERS, to_rows(customer_360))}")
print(f"orders written:    {write_batches(MERGE_ORDERS, to_rows(orders))}")

## 7. Read it back

The counts should match section 3. The second query is the point of the exercise: one hop
across `PLACED` answers what would be a self-join over the order table.

In [ ]:
COUNTS = """
MATCH (c:Customer) WITH count(c) AS customers
MATCH (o:Order)    WITH customers, count(o) AS orders
MATCH ()-[r:PLACED]->() RETURN customers, orders, count(r) AS placed
"""

TOP_CUSTOMERS = """
MATCH (c:Customer)-[:PLACED]->(o:Order)
RETURN c.full_name AS customer, c.tier AS tier, c.city AS city,
       count(o) AS orders, round(sum(o.amount), 2) AS spend
ORDER BY spend DESC
LIMIT 10
"""

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    counts = driver.execute_query(COUNTS, database_=NEO4J_DATABASE).records[0]
    top = driver.execute_query(TOP_CUSTOMERS, database_=NEO4J_DATABASE).records

print(f"in graph: {counts['customers']} customers, {counts['orders']} orders, {counts['placed']} PLACED")
display(spark.createDataFrame([record.data() for record in top]))

## 8. Cleanup, optional

Vocareum's teardown cannot reach Aura, so nothing removes this data when the lab ends.
Set `RUN_CLEANUP = True` in section 1 and re-run this cell to drop the nodes this
notebook created. It touches only `:Customer` and `:Order`, and leaves the constraints in
place for the next run.

In [ ]:
if RUN_CLEANUP:
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        summary = driver.execute_query(
            "MATCH (n) WHERE n:Customer OR n:Order DETACH DELETE n",
            database_=NEO4J_DATABASE,
        ).summary
    print(f"deleted {summary.counters.nodes_deleted} nodes")
else:
    print("RUN_CLEANUP is False, nothing deleted")